In [0]:
--crosswalk 
SELECT DISTINCT
  TRIM(region_name) AS source_region
FROM pondolens.dev_bronze.hospital
ORDER BY source_region;

SELECT DISTINCT
  REPLACE(TRIM(region_name), ' - ', ' ') AS source_region
FROM pondolens.dev_bronze.hospital
ORDER BY source_region;

DESCRIBE TABLE pondolens.dev_bronze.region_boundaries;

SELECT DISTINCT
  TRIM(adm1_psgc) AS region_psgc,
  TRIM(adm1_en) AS region_name
FROM pondolens.dev_bronze.region_boundaries
WHERE adm1_psgc IS NOT NULL
  AND adm1_en IS NOT NULL
ORDER BY region_name;

SELECT DISTINCT
  adm1_psgc,
  adm1_en
FROM pondolens.dev_bronze.region_boundaries
ORDER BY adm1_psgc;

WITH hospital_region_map AS (
  SELECT * FROM VALUES
    ('BARMM', '1900000000'),
    ('Cordillera Administrative Region', '1400000000'),
    ('MIMAROPA Region', '1700000000'),
    ('NCR - National Capital Region', '1300000000'),
    ('Region I - Ilocos Region', '0100000000'),
    ('Region II - Cagayan Valley', '0200000000'),
    ('Region III - Central Luzon', '0300000000'),
    ('Region IV-A CALABARZON', '0400000000'),
    ('Region IX - Zamboanga Peninsula', '0900000000'),
    ('Region V - Bicol Region', '0500000000'),
    ('Region VI - Western Visayas', '0600000000'),
    ('Region VII - Central Visayas', '0700000000'),
    ('Region VIII - Eastern Visayas', '0800000000'),
    ('Region X - Northern Mindanao', '1000000000'),
    ('Region XI - Davao Region', '1100000000'),
    ('Region XII - SOCCSKSARGEN', '1200000000'),
    ('Region XIII - Caraga', '1600000000')
  AS mapping(source_region, region_psgc)
),
region_reference AS (
  SELECT DISTINCT
    LPAD(TRIM(adm1_psgc), 10, '0') AS region_psgc,
    TRIM(adm1_en) AS standardized_region_name
  FROM pondolens.dev_bronze.region_boundaries
)
SELECT DISTINCT
  TRIM(h.region_name) AS source_region,
  r.standardized_region_name,
  m.region_psgc,
  CASE
    WHEN m.source_region IS NULL THEN 'MAPPING MISSING'
    WHEN r.region_psgc IS NULL THEN 'PSGC CODE NOT FOUND'
    ELSE 'MAPPED'
  END AS mapping_status
FROM pondolens.dev_bronze.hospital h
LEFT JOIN hospital_region_map m
  ON TRIM(h.region_name) = m.source_region
LEFT JOIN region_reference r
  ON m.region_psgc = r.region_psgc
ORDER BY source_region;



CREATE TABLE IF NOT EXISTS pondolens.dev_silver.shared_region_crosswalk (
  source_region STRING,
  region_code STRING
)
USING DELTA;

MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS target
USING (
  SELECT * FROM VALUES
    ('BARMM', '1900000000'),
    ('Cordillera Administrative Region', '1400000000'),
    ('MIMAROPA Region', '1700000000'),
    ('NCR - National Capital Region', '1300000000'),
    ('Region I - Ilocos Region', '0100000000'),
    ('Region II - Cagayan Valley', '0200000000'),
    ('Region III - Central Luzon', '0300000000'),
    ('Region IV-A CALABARZON', '0400000000'),
    ('Region IX - Zamboanga Peninsula', '0900000000'),
    ('Region V - Bicol Region', '0500000000'),
    ('Region VI - Western Visayas', '0600000000'),
    ('Region VII - Central Visayas', '0700000000'),
    ('Region VIII - Eastern Visayas', '0800000000'),
    ('Region X - Northern Mindanao', '1000000000'),
    ('Region XI - Davao Region', '1100000000'),
    ('Region XII - SOCCSKSARGEN', '1200000000'),
    ('Region XIII - Caraga', '1600000000')
  AS mappings(source_region, region_code)
) AS source
ON TRIM(target.source_region) = source.source_region
WHEN NOT MATCHED THEN
  INSERT (source_region, region_code)
  VALUES (source.source_region, source.region_code);


  SELECT source_region, region_code
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY source_region;


--clean no dash in region
SELECT DISTINCT
  CONCAT('[', TRIM(region_name), ']') AS bronze_region_name,
  CONCAT(
    '[',
    REGEXP_REPLACE(TRIM(region_name), '\\s*-\\s*', ' '),
    ']'
  ) AS cleaned_region_name
FROM pondolens.dev_bronze.hospital
WHERE region_name LIKE '%-%'
ORDER BY bronze_region_name;


MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS target
USING (
  SELECT * FROM VALUES
    ('NCR - National Capital Region', 'NCR National Capital Region', '1300000000'),
    ('Region I - Ilocos Region', 'Region I Ilocos Region', '0100000000'),
    ('Region II - Cagayan Valley', 'Region II Cagayan Valley', '0200000000'),
    ('Region III - Central Luzon', 'Region III Central Luzon', '0300000000'),
    ('Region IV-A CALABARZON', 'Region IV A CALABARZON', '0400000000'),
    ('Region IX - Zamboanga Peninsula', 'Region IX Zamboanga Peninsula', '0900000000'),
    ('Region V - Bicol Region', 'Region V Bicol Region', '0500000000'),
    ('Region VI - Western Visayas', 'Region VI Western Visayas', '0600000000'),
    ('Region VII - Central Visayas', 'Region VII Central Visayas', '0700000000'),
    ('Region VIII - Eastern Visayas', 'Region VIII Eastern Visayas', '0800000000'),
    ('Region X - Northern Mindanao', 'Region X Northern Mindanao', '1000000000'),
    ('Region XI - Davao Region', 'Region XI Davao Region', '1100000000'),
    ('Region XII - SOCCSKSARGEN', 'Region XII SOCCSKSARGEN', '1200000000'),
    ('Region XIII - Caraga', 'Region XIII Caraga', '1600000000')
  AS mappings(old_source_region, new_source_region, region_code)
) AS source
ON TRIM(target.source_region) = source.old_source_region
WHEN MATCHED THEN UPDATE SET
  target.source_region = source.new_source_region,
  target.region_code = source.region_code;


SELECT
  COUNT(*) AS matched_rows,
  COUNT(DISTINCT x.source_region) AS unique_source_regions,
  SUM(CASE WHEN x.source_region RLIKE '\\s-\\s' THEN 1 ELSE 0 END)
    AS labels_still_with_separator_dash,
  SUM(CASE WHEN x.region_code IS NULL OR x.region_code = '' THEN 1 ELSE 0 END)
    AS missing_region_codes
FROM (
  SELECT DISTINCT
    REGEXP_REPLACE(TRIM(region_name), '\\s*-\\s*', ' ') AS source_region
  FROM pondolens.dev_bronze.hospital
) h
JOIN pondolens.dev_silver.shared_region_crosswalk x
  ON h.source_region = x.source_region;

  ------------------------------------------------------------------


  


